# Analyse d'une campagne Tron

Ce notebook régénère tableaux, tests et figures à partir d'un fichier `results.csv` produit
par `tron run`. Il reprend les étapes de `tron analyze` pour une exploration interactive.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from tron.analysis import figures, load_results, paired_comparisons, summarize
from tron.analysis.regression import default_formula, fit_logit

RESULTS = Path("../results/pilote/results.csv")  # à adapter
OUT = RESULTS.parent / "notebook"
df = load_results(str(RESULTS))
print(f"{len(df)} parties, facteurs : {[c for c in df.columns if c.startswith('f_')]}")

## Taux de victoire par condition (intervalles de Wilson à 95 %)

In [ ]:
summary = summarize(df)
summary.head(20)

## H1 : rendements de la profondeur (paires de parties identiques, McNemar + Holm)

In [ ]:
paired_comparisons(df, "f_solo_depth")

## Figures

In [ ]:
for coalition_depth, sub in summary.groupby("f_coalition_depth"):
    path = figures.winrate_vs_depth(sub, OUT / f"winrate_cd{coalition_depth}.png")
    display(Image(filename=str(path)))
display(Image(filename=str(figures.heatmaps(summary, OUT / "heatmaps.png"))))

## Régression logistique (H1, H2, H3)

In [ ]:
formula = default_formula(df)
reg = fit_logit(df, formula)
print(reg.to_text())

## Profondeur exprimée en tours complets

`solo_depth_rounds` = plies ÷ nombre de joueurs : utile pour comparer les tailles de coalition.

In [ ]:
summarize(df, by=["f_k", "solo_depth_rounds"]).head(20)